# Urban Mobility Analytics MVP
## Notebook 01: Exploratory Data Analysis & Metadata Overview

This notebook serves as an interactive walkthrough of the primary data sources utilized in this MVP.

### Objectives:
1. **Load and inspect** raw or processed SUBE transaction datasets.
2. **Analyze the spatial context** using local shapefiles (Comunas / Barrios).
3. **Examine public transit network representation** from GTFS datasets.

---

### 1. Environment Setup & Configuration

In [1]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import yaml

# Load configuration parameters
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")
print("Processed directory:", config['paths']['processed_dir'])

ModuleNotFoundError: No module named 'matplotlib'

### 2. SUBE Demand Data Inspection (Transactions & Cards)

In [ ]:
# Load processed daily mobility metrics
metrics_path = os.path.join('..', config['sube']['mobility_metrics_output'])
metrics_df = pd.read_parquet(metrics_path)

print(f"Daily Mobility Metrics shape: {metrics_df.shape}")
metrics_df.head()

In [ ]:
# Brief statistical summary of transactions and active cards
metrics_df[['total_transacciones', 'total_tarjetas_activas', 'viajes_por_tarjeta']].describe()

In [ ]:
# Plotting overall transaction volume and active cards over time
plt.figure(figsize=(12, 5))
plt.plot(metrics_df['DIA_TRANSPORTE'], metrics_df['total_transacciones'], label='Total Transactions', color='#1f77b4')
plt.plot(metrics_df['DIA_TRANSPORTE'], metrics_df['total_tarjetas_activas'], label='Active Cards', color='#ff7f0e')
plt.title('Daily Public Transport Volume - SUBE (2025)')
plt.xlabel('Date')
plt.ylabel('Count')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3. Spatial Context & Shapefiles Exploration

In [ ]:
# Load administrative divisions (Comunas)
comunas_path = os.path.join('..', config['paths']['comunas_shapefile'])
if os.path.exists(comunas_path):
    comunas_gdf = gpd.read_file(comunas_path)
    print(f"Comunas CRS: {comunas_gdf.crs}")
    print(comunas_gdf.head(2))
else:
    print("Comunas shapefile not found.")

In [ ]:
# Quick visualization of administrative divisions
if 'comunas_gdf' in locals():
    fig, ax = plt.subplots(figsize=(8, 8))
    comunas_gdf.plot(ax=ax, edgecolor='black', color='#e2e8f0', alpha=0.6)
    for idx, row in comunas_gdf.iterrows():
        # Extract COMUNAS label identifier safely
        label_col = [c for c in comunas_gdf.columns if 'COMUNA' in c.upper()][0]
        plt.annotate(text=f"C{int(float(row[label_col]))}", 
                     xy=(row.geometry.centroid.x, row.geometry.centroid.y),
                     horizontalalignment='center', fontsize=9, fontweight='bold', color='#1e293b')
    plt.title('Buenos Aires Communes (Administrative Divisions)')
    plt.axis('off')
    plt.show()

### 4. Public Transit (GTFS) Network Integration

In [ ]:
# Load GTFS stops data
stops_path = os.path.join('..', config['gtfs']['stops_output'])
stops_df = pd.read_parquet(stops_path)
print(f"Stops shape: {stops_df.shape}")
stops_df.head(3)

In [ ]:
# Plotting stops over Comunas to visualize the geographic density
if 'comunas_gdf' in locals():
    # Convert stops to a GeoDataFrame
    stops_gdf = gpd.GeoDataFrame(
        stops_df, 
        geometry=gpd.points_from_xy(stops_df['stop_lon'], stops_df['stop_lat']),
        crs="EPSG:4326"
    )
    
    # Reproject comunas to EPSG:4326 if needed
    if comunas_gdf.crs != "EPSG:4326":
        comunas_gdf = comunas_gdf.to_crs("EPSG:4326")
        
    fig, ax = plt.subplots(figsize=(10, 10))
    comunas_gdf.plot(ax=ax, edgecolor='#94a3b8', color='#f1f5f9', alpha=0.8)
    stops_gdf.plot(ax=ax, markersize=3, color='#ef4444', alpha=0.6, label='GTFS Transit Stop')
    plt.title('Public Transit Stops Density across Communes')
    plt.legend()
    plt.axis('off')
    plt.show()